# Non instruction pretrain LLM finetuning on Domain Specific Dataset
* Notebook by Adam Lang
* Date: 3/19/2026

## Overview
* Fine-tuning a pre-trained LLM on a domain-specific dataset without instructions is more accurately termed Domain Adaptation (or Continued Pre-training) rather than standard Instruction-based Supervised Fine-Tuning (SFT).
* While both are supervised, the former focuses on updating knowledge, jargon, and style, while SFT focuses on teaching prompt-following behavior.

### Key Differences
* Non-Instruction Domain Adaptation:

1. Fine-tuning on raw domain text (e.g., medical papers, legal contracts) to teach the model new vocabulary and specialized knowledge.

2. Instruction SFT: Fine-tuning on prompt-response pairs to teach the model how to follow commands, format output, and act as an assistant.

### Why it's for Domain Adaptation
1. Specialized Knowledge: It embeds new, specialized information into the model weights, such as technical terminology or domain-specific styles.

2. Vocabulary Adaptation: The model learns to understand the nuances of a new field (e.g., medical, finance) without needing to follow commands in every training example.

3. No Instruction Structure: Unlike SFT, it uses conversational or raw text to improve reasoning within that field rather than changing how the model interacts with the user

## Recent paper
* See this recent paper by Lin et al, 2025. [SFT Doesn’t Always Hurt General Capabilities: Revisiting Domain-Specific Fine-Tuning in LLMs](https://arxiv.org/html/2509.20758v2)


## Libraries needed
- Transformers
- hugging face datasets
- accelerate
- bitsandbytes
- PEFT

# 0. Install

In [1]:
!pip install -U peft bitsandbytes transformers accelerate

In [2]:
!pip install -U trl

In [3]:
!pip install PyMuPDF

## Non-Instruction Finetuning Dataset

In [ ]:
# ## datasets
# dataset = load_dataset("HuggingFaceFW/fineweb")
# pubmed = load_dataset("ncbi/pubmed")
# dataset = load_dataset("datajuicer/the-pile-pubmed-abstracts-refined-by-data-juicer")
# dataset = load_dataset("open-llm-leaderboard/open_llm_corpus")
# owt = load_dataset("Skylion007/openwebtext")
# ds = load_dataset("armanc/scientific_papers")

## Instruction Finetuning Dataset

In [ ]:
https://huggingface.co/datasets/Amod/mental_health_counseling_conversations
https://huggingface.co/datasets/yahma/alpaca-cleaned
https://huggingface.co/datasets/Open-Orca/OpenOrca
https://huggingface.co/datasets/tatsu-lab/alpaca
https://huggingface.co/datasets/OpenAssistant/oasst1

## Preference Alignment Tuning Dataset

In [ ]:
https://huggingface.co/datasets/Anthropic/hh-rlhf
https://huggingface.co/datasets/argilla/ultrafeedback-binarized-preferences-cleaned
https://huggingface.co/datasets/xinlai/Math-Step-DPO-10k

# Prebuilt data from huggingface data hub

In [4]:
from datasets import Dataset, load_dataset

## load dataset: https://huggingface.co/datasets/roneneldan/TinyStories
dataset = load_dataset("roneneldan/TinyStories",
                       split="train")

In [5]:
## print data
print(dataset)

Dataset({
    features: ['text'],
    num_rows: 2119719
})


In [6]:
## rows
dataset[0]

{'text': 'One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.\n\nLily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."\n\nTogether, they shared the needle and sewed the button on Lily\'s shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.'}

# 1. Load custom data (non instruction data) for domain specific fine tuning
* Dataset location: https://github.com/sunnysavita10/Complete-LLM-Finetuning/blob/main/LLM%20Fine-Tuning-14-Train-LLMs-on-Your-PDF-Text-Data%20-Domain-Specific-Fine-Tuning-with-HuggingFace/Metformin.pdf

## Install PyPDF2
* Subtask:
* Install the PyPDF2 library, as it will be used for PDF text extraction instead of PyMuPDF (fitz).

* Reasoning: The subtask is to install the PyPDF2 library. I will use !pip install PyPDF2 in a code block to accomplish this.

In [8]:
import sys
!pip install PyPDF2
print("PyPDF2 installed successfully.")

PyPDF2 installed successfully.


Reasoning: The previous subtask of installing PyPDF2 has been completed. The next step, according to the main task, is to define a function to extract text from a PDF using PyPDF2 and then use it to extract text from 'Metformin.pdf'. I will first define the function and then call it with the specified PDF.



## Download PDF file

In [9]:
import os

# Original GitHub blob URL
github_blob_url = "https://github.com/sunnysavita10/Complete-LLM-Finetuning/blob/main/LLM%20Fine-Tuning-14-Train-LLMs-on-Your-PDF-Text-Data%20-Domain-Specific-Fine-Tuning-with-HuggingFace/Metformin.pdf"

# Convert to raw GitHub URL for direct download
raw_github_url = github_blob_url.replace(
    "github.com", "raw.githubusercontent.com"
).replace("/blob/", "/")

# Define the local path where the PDF will be saved
pdf_filename = "Metformin.pdf"
pdf_path = os.path.join("/content/", pdf_filename)

# Download the PDF using wget
!wget -O {pdf_path} {raw_github_url}

print(f"PDF downloaded to: {pdf_path}")

--2026-03-20 18:59:39--  https://raw.githubusercontent.com/sunnysavita10/Complete-LLM-Finetuning/main/LLM%20Fine-Tuning-14-Train-LLMs-on-Your-PDF-Text-Data%20-Domain-Specific-Fine-Tuning-with-HuggingFace/Metformin.pdf
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.110.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 57048 (56K) [application/octet-stream]
Saving to: ‘/content/Metformin.pdf’

/content/Metformin. 100%[===================>]  55.71K  --.-KB/s    in 0.005s  

2026-03-20 18:59:39 (10.3 MB/s) - ‘/content/Metformin.pdf’ saved [57048/57048]

PDF downloaded to: /content/Metformin.pdf


* Note: Older version of function. Fitz has too many dependencies so stopped using.

In [2]:
# import sys
# # Remove 'fitz' from sys.modules to force a fresh import
# if 'fitz' in sys.modules:
#     del sys.modules['fitz']

# import fitz ## load pdf

# print(f"Loaded fitz from: {fitz.__file__}")
# # print(f"fitz version: {fitz.__version__}") # Commenting this out as it previously failed

# ## function to extract text from pdf
# def extract_text_from_pdf(pdf_path):
#   text_blocks = []
#   ## open pdf file
#   with fitz.open(pdf_path) as doc:
#     ## iterate over and strip text into list
#     for page in doc:
#       text = page.get_text("text").strip()
#       if text:
#         text_blocks.append(text)
#   return text_blocks

Loaded fitz from: None


In [34]:
import PyPDF2

def extract_text_from_pdf_pypdf2(pdf_path):
    text_blocks = []
    with open(pdf_path, 'rb') as file:
        reader = PyPDF2.PdfReader(file)
        for page_num in range(len(reader.pages)):
            page = reader.pages[page_num]
            text = page.extract_text()
            if text:
                text_blocks.append(text.strip())
    return text_blocks


In [35]:
## load from path
pdf_texts = extract_text_from_pdf_pypdf2("/content/Metformin.pdf")

In [15]:
pdf_texts

['Metformin\n \nis\n \none\n \nof\n \nthe\n \nmost\n \nwidely\n \nprescribed\n \noral\n \nantihyperglycemic\n \nagents.\n \n \nIts\n \nprimary\n \nmechanism\n \nof\n \naction\n \ninvolves\n \nthe\n \nactivation\n \nof\n \nAMP-activated\n \nprotein\n \nkinase\n \n(AMPK)\n,\n \na\n \ncentral\n \nmetabolic\n \nregulator\n \nthat\n \npromotes\n \nglucose\n \nuptake\n \nand\n \nfatty\n \nacid\n \noxidation\n \nwhile\n \ninhibiting\n \nhepatic\n \ngluconeogenesis.\n \n \nBeyond\n \nits\n \nglycemic\n \ncontrol,\n \nMetformin\n \nhas\n \nbeen\n \nshown\n \nto\n \nimprove\n \ncardiovascular\n \noutcomes\n \nand\n \ndisplay\n \nanti-inflammatory\n \nproperties.\n \n \nRecent\n \nstudies\n \nalso\n \nsuggest\n \npotential\n \nanticancer\n \neffects\n \nthrough\n \ninhibition\n \nof\n \nthe\n \nmTOR\n \nsignaling\n \npathway\n \nand\n \nsuppression\n \nof\n \ntumor\n \nangiogenesis.\n \n \nClinical\n \ntrials\n \nhave\n \ndemonstrated\n \nthat\n \ncombining\n \nAtorvastatin\n \nwith\n \nEzetimibe

# 2. Chunk PDF
* Process we need to remember:

1. data collection
2. cleaning/filtering
3. splitting/chunking
4. tokenization
5. training (next-token prediction) --> model learns to predict next token in chunk

In [37]:
print(repr(pdf_texts[0][:500]))

'Metformin\n \nis\n \none\n \nof\n \nthe\n \nmost\n \nwidely\n \nprescribed\n \noral\n \nantihyperglycemic\n \nagents.\n \n \nIts\n \nprimary\n \nmechanism\n \nof\n \naction\n \ninvolves\n \nthe\n \nactivation\n \nof\n \nAMP-activated\n \nprotein\n \nkinase\n \n(AMPK)\n,\n \na\n \ncentral\n \nmetabolic\n \nregulator\n \nthat\n \npromotes\n \nglucose\n \nuptake\n \nand\n \nfatty\n \nacid\n \noxidation\n \nwhile\n \ninhibiting\n \nhepatic\n \ngluconeogenesis.\n \n \nBeyond\n \nits\n \nglycemic\n \ncontrol,\n \nMetformin\n \nhas\n \nbeen\n \nshown\n \nto\n \nimprove\n \ncardiovascular\n \noutcomes\n \nan'


In [38]:
import re

def split_paragraphs(pages):
    paragraphs = []
    for page_text in pages:
        # 1. Replace PARAGRAPH breaks first (the longer pattern)
        #    \n \n \n  = sentence/paragraph boundary
        text = page_text.replace('\n \n \n', '<PARA>')

        # 2. Then collapse WORD breaks (the shorter pattern)
        #    \n \n  = word boundary artifact from PyPDF2
        text = text.replace('\n \n', ' ')

        # 3. Catch any remaining stray newlines
        text = text.replace('\n', ' ')

        # 4. Clean up extra whitespace
        text = re.sub(r' +', ' ', text)

        # 5. Split on paragraph markers
        chunks = text.split('<PARA>')

        for chunk in chunks:
            clean = chunk.strip()
            if len(clean) > 30:
                paragraphs.append(clean)
    return paragraphs

paragraphs = split_paragraphs(pdf_texts)
print(f"Number of paragraphs found: {len(paragraphs)}")
for i, p in enumerate(paragraphs):
    print(f"\nParagraph {i+1}: {p[:200]}...")

Number of paragraphs found: 15

Paragraph 1: Metformin is one of the most widely prescribed oral antihyperglycemic agents....

Paragraph 2: Its primary mechanism of action involves the activation of AMP-activated protein kinase (AMPK) , a central metabolic regulator that promotes glucose uptake and fatty acid oxidation while inhibiting he...

Paragraph 3: Beyond its glycemic control, Metformin has been shown to improve cardiovascular outcomes and display anti-inflammatory properties....

Paragraph 4: Recent studies also suggest potential anticancer effects through inhibition of the mTOR signaling pathway and suppression of tumor angiogenesis....

Paragraph 5: Clinical trials have demonstrated that combining Atorvastatin with Ezetimibe results in significant reductions in low-density lipoprotein cholesterol (LDL-C) levels compared to monotherapy....

Paragraph 6: Ezetimibe acts by inhibiting the Niemann–Pick C1-like 1 (NPC1L1) transporter in the intestinal wall, reducing cholesterol abs

In [39]:
## Display the chunked paragraphs
if paragraphs:
    for i, p in enumerate(paragraphs):
        print(f"Paragraph {i+1}: {p[:150]}...")
else:
    print("No paragraphs found. Checking raw text structure...")
    if pdf_texts:
        print(repr(pdf_texts[0][:200]))

Paragraph 1: Metformin is one of the most widely prescribed oral antihyperglycemic agents....
Paragraph 2: Its primary mechanism of action involves the activation of AMP-activated protein kinase (AMPK) , a central metabolic regulator that promotes glucose u...
Paragraph 3: Beyond its glycemic control, Metformin has been shown to improve cardiovascular outcomes and display anti-inflammatory properties....
Paragraph 4: Recent studies also suggest potential anticancer effects through inhibition of the mTOR signaling pathway and suppression of tumor angiogenesis....
Paragraph 5: Clinical trials have demonstrated that combining Atorvastatin with Ezetimibe results in significant reductions in low-density lipoprotein cholesterol ...
Paragraph 6: Ezetimibe acts by inhibiting the Niemann–Pick C1-like 1 (NPC1L1) transporter in the intestinal wall, reducing cholesterol absorption, while Atorvastat...
Paragraph 7: The dual mechanism provides an additive lipid-lowering effect, particularly benefici

# 2. Map data to text column

In [40]:
## iterate on chunk and append to text column
data = [{"text": p} for p in paragraphs]
print(f"Prepared {len(data)} rows for the dataset.")

Prepared 15 rows for the dataset.


In [41]:
## data
print(data)

[{'text': 'Metformin is one of the most widely prescribed oral antihyperglycemic agents.'}, {'text': 'Its primary mechanism of action involves the activation of AMP-activated protein kinase (AMPK) , a central metabolic regulator that promotes glucose uptake and fatty acid oxidation while inhibiting hepatic gluconeogenesis.'}, {'text': 'Beyond its glycemic control, Metformin has been shown to improve cardiovascular outcomes and display anti-inflammatory properties.'}, {'text': 'Recent studies also suggest potential anticancer effects through inhibition of the mTOR signaling pathway and suppression of tumor angiogenesis.'}, {'text': 'Clinical trials have demonstrated that combining Atorvastatin with Ezetimibe results in significant reductions in low-density lipoprotein cholesterol (LDL-C) levels compared to monotherapy.'}, {'text': 'Ezetimibe acts by inhibiting the Niemann–Pick C1-like 1 (NPC1L1) transporter in the intestinal wall, reducing cholesterol absorption, while Atorvastatin inhi

# 3. Convert to hugging face data structure
- compatabile for data loading

In [42]:
from datasets import Dataset
## hf dataset structure
dataset = Dataset.from_list(data)
print(dataset)

Dataset({
    features: ['text'],
    num_rows: 15
})


In [43]:
## first row
print(dataset[0])

{'text': 'Metformin is one of the most widely prescribed oral antihyperglycemic agents.'}


# 4. Select Model
* model card: https://huggingface.co/TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T
  *  TinyLlama model trained for ~1.4M steps over 3 trillion tokens, captured midway before final convergence.

In [44]:
## need to select model based on GPU and VRAM
# model_name = "meta-llama/Llama-2-7b-hf"
# model_name = "meta-llama/Llama-3.1-8b"
model_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

In [45]:
## load libraries for fine-tuning
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
)

# 5. Load Tokenizer

In [ ]:
## tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
tokenizer

In [ ]:
tokenizer.vocab_size

In [ ]:
tokenizer.all_special_tokens

## Set pad_token = eos_token -- why?

* `eos_token` → tells the model where to stop.

* `pad_token` → only relevant for batches.

* Setting `pad_token_id = eos_token_id` → safe default for open-ended text generation.
* [Source](https://medium.com/@khalandar.mokula/understanding-eos-token-and-pad-token-in-model-generate-config-huggingface-transformers-fef6845ed92a)



In [ ]:
## pad_token = eos_token
if tokenizer.pad_token is None:
  tokenizer.pad_token = tokenizer.eos_token

# 6. Preprocess data

In [ ]:
## tokenize and process data
def tokenize_fn(examples):
  ## apply tokenizer
  tokens = tokenizer(examples["text"],
                     truncation=True,
                     padding="max_length",
                     max_length=512)
  ## label2id
  tokens["labels"] = tokens["input_ids"].copy()
  return tokens

## Map data
* We are removing the "text" column at end because we only want:
1. input_ids
2. attention_mask
3. labels

In [ ]:
## map data
tokenized = dataset.map(tokenize_fn, batched=True, remove_columns=["text"])

In [ ]:
tokenized

# 7. Load Model

In [ ]:
## load the model
model = AutoModelForCausalLM.from_pretrained(model_name)
model

# 8. Training Arguments
- output_dir (save model checkpoints)
- num_train_epochs (training cycles, more=longer)
- per_device_train_batch_size (batch per GPU, affects speed & memory)
- save_steps (save frequency)
- save_total_limit (keep last N)
- logging_steps (log every X steps)
- learning_rate (weight update rate)
- fp16 (mixed precision)
- report_to (logging destination)

In [ ]:
## training args
training_args = TrainingArguments(
    output_dir="/.llama-pharma-domain-ft",
    num_train_epochs=2,
    per_device_train_batch_size=2,
    save_steps=500,
    save_total_limit=2,
    logging_steps=50,
    learning_rate=2e-5,
    fp16=True,
    report_to="none",
)

In [ ]:
## training arg help
from transformers import TrainingArguments
help(TrainingArguments)

# 9. Define Trainer

In [ ]:
## trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized,
)

# Now we have to setup Partial Fine-Tuning -- dont fine tune entire model its HUGE!
* There are 2 methods we can try (out of many):

1. Freeze some layers and finetune unfreeze layer (old CNN and BERT style method)

2. LoRA - append some external weights to the already trained pretrain weight.

## LoRA fine tuning

In [46]:
## set up garbage collector
import torch, gc
gc.collect()
torch.cuda.empty_cache()

In [47]:
## imports
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset

In [48]:
## load model (same one)
model = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

In [49]:
## tokenizer
tokenizer = AutoTokenizer.from_pretrained(model)

config.json:   0%|          | 0.00/560 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/776 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

In [50]:
## assign eos token to pad token
if tokenizer.pad_token is None:
  tokenizer.pad_token = tokenizer.eos_token

In [51]:
## map dataset label2id
def tokenize_fn(examples):
  tokens = tokenizer(
      examples["text"],
      truncation=True,
      padding="max_length",
      max_length=512,
  )
  tokens["labels"] = tokens["input_ids"].copy()
  return tokens

In [52]:
## tokenize
tokenized = dataset.map(tokenize_fn, batched=True)
tokenized

Map:   0%|          | 0/15 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 15
})

In [54]:
from transformers import BitsAndBytesConfig

## Define quantization config
quantization_config = BitsAndBytesConfig(
    load_in_8bit=True
)

## load quantized model correctly
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto"
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/129 [00:00<?, ?B/s]

In [55]:
## LoRA config
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8, # rank of matrix
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"], # attention layer
    lora_dropout=0.05,
    bias="none",
)

In [56]:
## load peft model
q_lora_model = get_peft_model(model, lora_config)
q_lora_model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(32000, 2048)
        (layers): ModuleList(
          (0-21): 22 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear8bitLt(
                (base_layer): Linear8bitLt(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Li

In [57]:
## Training args
args = TrainingArguments(
    output_dir="./tinyllama-lora",
    num_train_epochs=5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=20,
    save_total_limit=1,
    report_to="none",
)

In [59]:
## define trainer
trainer = Trainer(
    model=q_lora_model, ## quantized model
    args=args,
    train_dataset=tokenized,
)

In [60]:
## no train the quantized model
trainer.train()

/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Step,Training Loss


TrainOutput(global_step=10, training_loss=9.470032501220704, metrics={'train_runtime': 42.7609, 'train_samples_per_second': 1.754, 'train_steps_per_second': 0.234, 'total_flos': 238611175833600.0, 'train_loss': 9.470032501220704, 'epoch': 5.0})

In [61]:
## get trained model checkpoint
model_path = "/content/tinyllama-lora/checkpoint-10"

### Load trained model

In [62]:
## trained lora (qlora)
trained_model = AutoModelForCausalLM.from_pretrained(model_path, device_map="auto")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/88 [00:00<?, ?it/s]

### Prompt Test

In [63]:
prompt = "Clinical trials demonstrated that combining Atorvastatin with Ezetimibe"

In [64]:
## tokenize prompt
inputs = tokenizer(prompt,
                   return_tensors="pt").to("cuda")

In [67]:
## generate output
outputs = trained_model.generate(
    **inputs,
    max_new_tokens=200,
    temperature=0.8,
    top_p=0.9, ## usually dont change top_p at same time as temp
    do_sample=True,
    repetition_penalty=1.1,
)

Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [68]:
print("\nModel Output:\n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


Model Output:

Clinical trials demonstrated that combining Atorvastatin with Ezetimibe improved the lipid profile, whereas ezetimibe alone was associated with a significant increase in LDL cholesterol.
Liquid and Oral Atorvastatin
Atorvastatin is available as an oral liquid for children 3 to 12 years of age (0.5 mg/day) and as an oral tablet for adults and pediatric patients (2.5 mg twice daily). A single dose of Atorvastatin has been shown to be well tolerated and effective in reducing LDL-C. The most common adverse events were headache and muscle pain. In some cases these may have been due to hypersensitivity reactions.
The recommended starting dosage of Atorvastatin for pediatric patients who weigh at least 11 kg is 0.5 mg/kg per day (administered in divided doses) administer


# Summary
* We can see the model is capable of generating a "domain specific" response but it lacks the ability to give the true tone, structure, or linguistic qualities that we really want in a response.
* That is where Step 2 of SFT comes in which is Instruction Fine Tuning. SFT ("normal fine tuning") gives us the domain knowledge but its usually not enough.